In [15]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path(r"C:\Users\skous\Super-League-odds-research")
SCOTLAND_RESEARCH = REPO_ROOT / "scotland_research"

assert SCOTLAND_RESEARCH.exists(), SCOTLAND_RESEARCH
sys.path.insert(0, str(SCOTLAND_RESEARCH))

from data.load_model_dataset import load_dataset
from evaluation.report import write_evaluation_outputs
from evaluation.walk_forward import run_walk_forward
from models import models_for_individual_feature_removal_test

ImportError: cannot import name 'models_for_individual_feature_removal_test' from 'models' (C:\Users\skous\Super-League-odds-research\scotland_research\models\__init__.py)

In [7]:
MODEL_DATASET = (REPO_ROOT/"data"/"processed"/"scotland"/"scotland_model_dataset.csv")
OUTPUT_DIR = (REPO_ROOT/"artifacts"/"scotland_player_feature_removal")

MODEL_DATASET, OUTPUT_DIR

(WindowsPath('C:/Users/skous/Super-League-odds-research/data/processed/scotland/scotland_model_dataset.csv'),
 WindowsPath('C:/Users/skous/Super-League-odds-research/artifacts/scotland_player_feature_removal'))

In [8]:
def difference_from_full_model(
    #Comparison with the full player model

    metrics: pd.DataFrame,
    group_column: str | None = None) -> pd.DataFrame:
    compared = metrics.copy()
    full_rows = compared[compared["model"].eq(FULL_PLAYER_MODEL_NAME)]

    if group_column is None:
        if len(full_rows) != 1:
            raise ValueError("Expected exactly one overall full-player-model row")
        compared["full_model_log_loss"] = full_rows["log_loss"].iloc[0]

    else:
        full_by_group = full_rows.set_index(group_column)["log_loss"]
        if full_by_group.index.duplicated().any():
            raise ValueError(f"Multiple full-player-model rows for {group_column}")
        compared["full_model_log_loss"] = compared[group_column].map(full_by_group)

        if compared["full_model_log_loss"].isna().any():
            raise ValueError(f"A {group_column} group has no full-player-model row")

    compared["log_loss_vs_full_player_model"] = (
        compared["log_loss"] - compared["full_model_log_loss"]
    )

    explanations = []
    for model, difference in zip(
        compared["model"],
        compared["log_loss_vs_full_player_model"],
        strict=True,
    ):
        if model == "closing_market":
            explanations.append("closing market reference")
        elif model == FULL_PLAYER_MODEL_NAME:
            explanations.append("full player model reference")
        elif difference > 0:
            explanations.append("removed group helped the full model")
        elif difference < 0:
            explanations.append("removed group hurt the full model")
        else:
            explanations.append("removed group made no difference")

    compared["result"] = explanations
    return compared

In [10]:
# Load data
dataset = load_dataset(MODEL_DATASET)
dataset.shape

(1121, 46)

In [14]:
# Run walk forward
result = run_walk_forward(dataset, models_for_player_feature_removal_test())
result.overall_metrics

,model,out_of_sample_matches,log_loss,brier_score,accuracy,closing_market_log_loss,log_loss_vs_closing_market,log_loss_relative_to_closing_market,market_comparison
0,closing_market,665,0.917761,0.538017,0.568421,0.917761,0.000000,0.000000,baseline
1,market_without_history_coverage,665,0.928840,0.541828,0.584962,0.917761,0.011079,0.012072,within_2_percent
2,market_without_recent_experience,665,0.932632,0.544551,0.565414,0.917761,0.014871,0.016204,within_2_percent
3,market_without_ratings,665,0.932692,0.544290,0.575940,0.917761,0.014931,0.016269,within_2_percent
4,market_without_shooting,665,0.936445,0.547197,0.565414,0.917761,0.018684,0.020359,worse_by_more_than_2_percent
5,market_without_defending,665,0.936939,0.546680,0.562406,0.917761,0.019178,0.020897,worse_by_more_than_2_percent
6,market_without_chance_creation,665,0.937584,0.547358,0.568421,0.917761,0.019823,0.021599,worse_by_more_than_2_percent
7,market_plus_all_player_features,665,0.937831,0.547318,0.559398,0.917761,0.020070,0.021868,worse_by_more_than_2_percent
